In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_ENDPOINT = os.getenv("LANGSMITH_ENDPOINT")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "true")
HF_TOKEN = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [3]:
from langchain_core.documents import Document

document = [
    Document(
        page_content="MCP is an open protocol that standardizes how applications provide context to LLMs. Think of MCP like a USB-C port for AI applications. Just as USB-C provides a standardized way to connect your devices to various peripherals and accessories, MCP provides a standardized way to connect AI models to different data sources and tools.",
        metadata={"source": "https://modelcontextprotocol.io/introduction"},
    ),
    Document(page_content="Hello, world!", metadata={"source": "https://example.com"}),
    Document(
        page_content="A2A is an open protocol that complements Anthropic's Model Context Protocol (MCP), which provides helpful tools and context to agents. Drawing on Google's internal expertise in scaling agentic systems, we designed the A2A protocol to address the challenges we identified in deploying large-scale, multi-agent systems for our customers. A2A empowers developers to build agents capable of connecting with any other agent built using the protocol and offers users the flexibility to combine agents from various providers. Critically, businesses benefit from a standardized method for managing their agents across diverse platforms and cloud environments. We believe this universal interoperability is essential for fully realizing the potential of collaborative AI agents.",
        metadata={
            "source": "https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/"
        },
    ),
]

In [4]:
document

[Document(metadata={'source': 'https://modelcontextprotocol.io/introduction'}, page_content='MCP is an open protocol that standardizes how applications provide context to LLMs. Think of MCP like a USB-C port for AI applications. Just as USB-C provides a standardized way to connect your devices to various peripherals and accessories, MCP provides a standardized way to connect AI models to different data sources and tools.'),
 Document(metadata={'source': 'https://example.com'}, page_content='Hello, world!'),
 Document(metadata={'source': 'https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/'}, page_content="A2A is an open protocol that complements Anthropic's Model Context Protocol (MCP), which provides helpful tools and context to agents. Drawing on Google's internal expertise in scaling agentic systems, we designed the A2A protocol to address the challenges we identified in deploying large-scale, multi-agent systems for our customers. A2A empowers developers t

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama3-70b-8192")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001C128F1A270>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C128F1AE40>, model_name='llama3-70b-8192', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
)

In [7]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    document, embeddings, persist_directory="chroma_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [8]:
vector_store.similarity_search("What is MCP?", k=2)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(id='1c768708-2cdf-4af0-80e9-a106915a3712', metadata={'source': 'https://modelcontextprotocol.io/introduction'}, page_content='MCP is an open protocol that standardizes how applications provide context to LLMs. Think of MCP like a USB-C port for AI applications. Just as USB-C provides a standardized way to connect your devices to various peripherals and accessories, MCP provides a standardized way to connect AI models to different data sources and tools.'),
 Document(id='73bde576-9166-4615-8370-0d91b6d13536', metadata={'source': 'https://modelcontextprotocol.io/introduction'}, page_content='MCP is an open protocol that standardizes how applications provide context to LLMs. Think of MCP like a USB-C port for AI applications. Just as USB-C provides a standardized way to connect your devices to various peripherals and accessories, MCP provides a standardized way to connect AI models to different data sources and tools.')]

In [9]:
await vector_store.asimilarity_search("What is MCP?", k=1)

[Document(id='1c768708-2cdf-4af0-80e9-a106915a3712', metadata={'source': 'https://modelcontextprotocol.io/introduction'}, page_content='MCP is an open protocol that standardizes how applications provide context to LLMs. Think of MCP like a USB-C port for AI applications. Just as USB-C provides a standardized way to connect your devices to various peripherals and accessories, MCP provides a standardized way to connect AI models to different data sources and tools.')]

In [10]:
await vector_store.asimilarity_search("Hello What?", k=1)

[Document(id='66aeb7a0-45a9-43d1-b2e7-30ab0ce53e5d', metadata={'source': 'https://example.com'}, page_content='Hello, world!')]

In [11]:
await vector_store.asimilarity_search("What is A2A?", k=1)

[Document(id='57a1681d-d09a-4475-99c8-4b2d920a3fb5', metadata={'source': 'https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/'}, page_content="A2A is an open protocol that complements Anthropic's Model Context Protocol (MCP), which provides helpful tools and context to agents. Drawing on Google's internal expertise in scaling agentic systems, we designed the A2A protocol to address the challenges we identified in deploying large-scale, multi-agent systems for our customers. A2A empowers developers to build agents capable of connecting with any other agent built using the protocol and offers users the flexibility to combine agents from various providers. Critically, businesses benefit from a standardized method for managing their agents across diverse platforms and cloud environments. We believe this universal interoperability is essential for fully realizing the potential of collaborative AI agents.")]

In [12]:
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vector_store.similarity_search).bind(k=1)
retriever.batch(
    [
        "What is MCP?",
        "Hello What?",
        "What is A2A?",
    ]
)

[[Document(id='1c768708-2cdf-4af0-80e9-a106915a3712', metadata={'source': 'https://modelcontextprotocol.io/introduction'}, page_content='MCP is an open protocol that standardizes how applications provide context to LLMs. Think of MCP like a USB-C port for AI applications. Just as USB-C provides a standardized way to connect your devices to various peripherals and accessories, MCP provides a standardized way to connect AI models to different data sources and tools.')],
 [Document(id='66aeb7a0-45a9-43d1-b2e7-30ab0ce53e5d', metadata={'source': 'https://example.com'}, page_content='Hello, world!')],
 [Document(id='57a1681d-d09a-4475-99c8-4b2d920a3fb5', metadata={'source': 'https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/'}, page_content="A2A is an open protocol that complements Anthropic's Model Context Protocol (MCP), which provides helpful tools and context to agents. Drawing on Google's internal expertise in scaling agentic systems, we designed the A2A proto

In [13]:
vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
    return_source_documents=True,
)

retriever.batch(
    [
        "What is MCP?",
        "Hello What?",
        "What is A2A?",
    ]
)

[[Document(id='1c768708-2cdf-4af0-80e9-a106915a3712', metadata={'source': 'https://modelcontextprotocol.io/introduction'}, page_content='MCP is an open protocol that standardizes how applications provide context to LLMs. Think of MCP like a USB-C port for AI applications. Just as USB-C provides a standardized way to connect your devices to various peripherals and accessories, MCP provides a standardized way to connect AI models to different data sources and tools.')],
 [Document(id='66aeb7a0-45a9-43d1-b2e7-30ab0ce53e5d', metadata={'source': 'https://example.com'}, page_content='Hello, world!')],
 [Document(id='57a1681d-d09a-4475-99c8-4b2d920a3fb5', metadata={'source': 'https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/'}, page_content="A2A is an open protocol that complements Anthropic's Model Context Protocol (MCP), which provides helpful tools and context to agents. Drawing on Google's internal expertise in scaling agentic systems, we designed the A2A proto

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """Answer this question using the provided context only.
{question}
Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

response = rag_chain.invoke("What is MCP?")
print(response.content)

According to the provided context, MCP is an open protocol that standardizes how applications provide context to LLMs (Large Language Models).


In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """Answer this question using the provided context only.
{question}
Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

response = rag_chain.invoke("What is A2A?")
print(response.content)

According to the provided context, A2A is an open protocol that enables agent interoperability, allowing agents built using the protocol to connect with any other agent, and providing a standardized method for managing agents across diverse platforms and cloud environments.
